In [16]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain.text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain.retrievers import BM25Retriever, EnsembleRetriever

from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

from langchain_groq import ChatGroq


# Load environment variables
load_dotenv()


# 1. Load documents
def load_documents(file_path):
    loader = TextLoader(file_path)
    return loader.load()


# 2. Split documents
def split_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100
    )
    return splitter.split_documents(documents)


# 3. Create vector database
def create_vector_db(chunks):
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vector_db = FAISS.from_documents(chunks, embeddings)
    return vector_db, embeddings


# 4. Hybrid Retriever (Dense + Sparse)
def create_retriever(chunks, vector_db):
    # Dense (semantic search)
    dense_retriever = vector_db.as_retriever(search_kwargs={"k": 4})

    # Sparse (keyword search)
    bm25 = BM25Retriever.from_documents(chunks)
    bm25.k = 4

    # Combine both
    retriever = EnsembleRetriever(
        retrievers=[bm25, dense_retriever],
        weights=[0.5, 0.5]
    )

    return retriever


# 5. QA System with Groq
def create_qa_system(retriever):
    llm = ChatGroq(
        model_name="llama3-8b-8192",  # fast + free
        temperature=0
    )

    prompt = PromptTemplate(
        input_variables=["context", "question"],
        template="""
You are an expert assistant.

Answer ONLY using the context below.
If the answer is not in context, say "I don't know".

Context:
{context}

Question:
{question}

Answer:
"""
    )

    qa = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={"prompt": prompt}
    )

    return qa


# MAIN FUNCTION
def main():
    file_path = r"D:\rag-project\data\text_files\sample.txt"

    # Pipeline
    docs = load_documents(file_path)
    chunks = split_documents(docs)

    vector_db, embeddings = create_vector_db(chunks)

    retriever = create_retriever(chunks, vector_db)

    qa = create_qa_system(retriever)

    # Query loop
    while True:
        query = input("\nAsk a question (or 'exit'): ")

        if query.lower() == "exit":
            break

        result = qa({"query": query})

        print("\n🧠 Answer:\n", result["result"])
        print("\n📄 Sources:\n", result["source_documents"])


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'langchain.text_splitters'